# Classification of Astronomical Events using CNNs (I)
```Authors: Paul Alvarez updated: 20260725```

This study implements and evaluates various computer vision models to classify astronomical events from the Zwicky Transient Facility (ZTF) survey. The main objective is to perform real-bogus classification using 63×63 pixel triplet images (science, reference, and difference) and analyze its performance in a reduced sample size. Additionally, it extends to multiclass classification (transient, periodic, stochastic) and includes transfer learning experiments using MobileNetV2 and Braai.


This first notebook covers the construction of the dataset: Collection and download of stamps via the ALeRCE/ZTF API, alongside normalization, handling of corrupted or missing data, and binary labeling.

## Table of contents:
* [Required libraries](#Required-libraries)
* [Introduction](#Introduction)
* [Dataset construction](#Dataset)
* [Download stamps](#initial-naive-approach)
* [References](#References)

## Required libraries

In [7]:
from alerce.core import Alerce
from astropy.io import fits
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import os
import time
import sqlalchemy as sa
import requests
import pandas as pd

## Introduction

Astronomy is currently booming in the era of Big Data. Surveys like the Zwicky Transient Facility (ZFT) utilize the Samuel Oschin Telescope at Palomar Observatory to detect transient phenomena in the night sky (such as supernovae, asteroids, AGN). The volume of generated data is inhumanly analyzable; manual inspection and classification of each observation is beyond our capacity. In recent years, with the emergence of models and brokers like Braai, Deep-Hits, and ALeRCE, convolutional neural networks (CNNs) have become essential tools and a rapidly growing area of exploration at the intersection of artificial intelligence and astronomy.

To process these observational data, the CNN receives astronomical astronomical "stamps". These consist of a triplet of images: recent observation image (Science), a historical baseline image of the same region (Reference), and the Difference image, which is obtained by subtracting the Reference image from Science. For instance, if a new point source appears in Science that was absent in Reference, it will stand out in the Difference image. The neural network uses this change in the image's distribution to determine whether a transient event is present.

Alert brokers such as ALeRCE are employing various pipelines, combining autoated algorithms with expert visual inspection to construct their final classification, however, they rely on large-scale datasets for training. Consequently, this project addresses the challenge of evaluating CNN performance in a data-limited regime. The primary objective is to demonstrate that effective, viable classifier can be built using a small dataset (~5000 stamps), while exploring the impact of techniques such as data augmentation or transfer learning.

## Dataset construction

We connect to the ALeRCE API, from there we can extract information on astronomical events to build the database.

In [8]:
# Connect to the ALeRCE API according to their instructions
client = Alerce()

url = "https://raw.githubusercontent.com/alercebroker/usecases/master/alercereaduser_v4.json"
params = requests.get(url).json()['params']

engine = sa.create_engine(f"postgresql+psycopg2://{params['user']}:{params['password']}@{params['host']}/{params['dbname']}")
engine.begin()

We generated a series of queries to construct the master dataset. First, we queried for confirmed real transient events and bogus detections (noise). Importantly, each query was constructed to ensure internal class balance, mitigating potential issues associated with class imbalance, such as majority class bias.

In [9]:
# API queries to obtain data
query = """
(
    SELECT DISTINCT ON (oid) oid, class_name
    FROM probability
    WHERE classifier_name = 'stamp_classifier'
      AND probability >= 0.76
      AND class_name = 'SN'
    LIMIT 1250
)
UNION ALL
(
    SELECT DISTINCT ON (oid) oid, class_name
    FROM probability
    WHERE classifier_name = 'stamp_classifier'
      AND probability >= 0.90
      AND class_name = 'VS'
    LIMIT 1250
);
"""

df_real = pd.read_sql_query(query, engine)

In [10]:
# API queries to obtain data
query = """
SELECT DISTINCT ON (o.oid)
       o.oid,
       p.class_name
FROM object o
JOIN probability p
ON o.oid = p.oid
WHERE p.classifier_name = 'stamp_classifier'
  AND p.class_name = 'bogus'
ORDER BY o.oid, p.probability DESC
LIMIT 2500;  
"""

df_bogus = pd.read_sql_query(query, engine)

With both queries, real events (CV and SN) and bogus, we will build our dataset, where each valid observation will be assigned class 1 (real) and bogus will be class 0 (false).

In [11]:
# df_real and df_bogus are the results of each query; we combine them to form our dataset
df_ztf = pd.concat([df_real, df_bogus], ignore_index=True)
df_ztf['label'] = df_ztf['class_name'].apply(lambda x: 0 if x=='bogus' else 1)
df_ztf = df_ztf.sample(frac=1, random_state=42).reset_index(drop=True) # no data leakage due to label position

df_ztf["class_name"].value_counts(normalize=True) * 100

class_name
bogus    50.0
VS       25.0
SN       25.0
Name: proportion, dtype: float64

In [12]:
# Save CSV of labels with only the oid and its label to apply CNN's
DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
save_path = os.path.join(DATA_DIR, "labels_real_vs_bogus.csv")

df_ztf[['oid','label']].to_csv(save_path, index=False)
df_ztf.head()

,oid,class_name,label
0,ZTF17aabcraf,VS,1
1,ZTF17aaaaaiu,bogus,0
2,ZTF17aaaaanw,bogus,0
3,ZTF18aazdgfr,SN,1
4,ZTF18aaxjvmh,SN,1


##  Download stamps